## Code Interpreter Tool with Databricks Agents

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Create and Execute the Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp
from databricks.sdk import WorkspaceClient

client = AsyncOpenAI(
    api_key=token,
    base_url=f"{workspace_host}/serving-endpoints"
)

model = OpenAIChatCompletionsModel(
    model="databricks-claude-sonnet-4-5",
    openai_client=client
)

# 3. Connect to the UC Function MCP Server
mcp_server_url = (
    f"{workspace_host}/api/2.0/mcp/functions/"
    f"system/ai/python_exec"
)

async with MCPServerStreamableHttp(
    name="code-interpreter",
    params={
        "url": mcp_server_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        }
    }
) as code_interpreter:
    # Don't send Agents SDK traces to OpenAI
    set_tracing_disabled(True)
    # 4. Create Agent
    agent = Agent(
        name="Coding agent",
        instructions=(
            "You are a helpful coding assistant. "
            "Use the python_exec tool to run code."
        ),
        model=model,
        mcp_servers=[code_interpreter],
    )
    
    # 5. Run Agent
    result = await Runner.run(
        agent,
        "Calculate the first 10 Fibonacci numbers"
    )

    print(result.final_output)